# Stage 4 — Fix the folds

**The question:** how will every future number in this project be measured?

This notebook produces the most important file in the project. Every score
anyone reports from here to the end is measured against it, which means it can
only be created once. If two people use different splits, their numbers cannot
be compared, and neither can a number from today against one from next month.

A **fold** is one of five equal parts we cut the data into. We train five
models; model 3 trains on folds 0, 1, 2, 4 and is tested on fold 3. Every slide
ends up predicted exactly once by a model that never saw it, and pooling those
predictions gives one honest score over the whole dataset.

Two things have to be true of the split, at the same time:

- **Grouped.** Duplicate slides must land in the same fold. If a slide is in
  training and its near-twin is in validation, the model has already seen the
  answer and the score comes out better than the truth.
- **Stratified.** Each fold should hold roughly the same mix of grades, and the
  same mix of hospitals. Otherwise one fold scores differently from another for
  reasons that have nothing to do with the model.

Those two pull against each other, because a duplicate group has to move as one
lump and that limits how evenly things can be spread. `StratifiedGroupKFold`
does its best at both; our job is to check the result rather than trust it.

In [1]:
"""Imports.

glob    finds files by pattern; used on Kaggle to locate the outputs of earlier
        notebooks, which arrive as attached datasets.
os      builds file paths and checks whether a file exists on disk.
numpy   holds the fold number assigned to each slide.
pandas  loads the three input tables, joins them, and reads and writes the
        fold file.
StratifiedGroupKFold (scikit-learn) splits the slides into folds so that
        duplicate groups stay together and each fold gets a similar mix of
        grades and hospitals.
"""

import glob
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

In [2]:
"""Paths and settings.

Where things are read from
--------------------------
1. ON_KAGGLE is True inside a Kaggle notebook, where /kaggle/input exists.
2. DATA_DIR holds the competition data. Kaggle mounts it read-only under
   /kaggle/input; locally it is ../data.
3. find_input(name) locates a file made by an earlier notebook. It looks in
   local_dir first, which is DERIVED_DIR unless told otherwise. On Kaggle every
   notebook runs separately, so earlier output usually arrives as an attached
   dataset instead; in that case it searches every attached dataset at the top
   level and inside derived/ and data/ folders. It prints where it found the
   file. If the file is nowhere, it stops with a clear error, unless
   required=False, in which case it returns None.
4. The three inputs, each found with find_input:
   - INVENTORY_PATH, from notebook 1: each slide's grade and hospital;
   - DUPLICATE_GROUPS_PATH, from notebook 3: each slide's duplicate group;
   - COLOUR_STATS_PATH, from notebook 2: each slide's ink score, used only for
     the pen check near the end, never for splitting.

Where things are written to
---------------------------
5. DERIVED_DIR is where the project's derived files live. On Kaggle it is
   /kaggle/working/derived/, because /kaggle/input is read-only.
6. FOLDS_DIR and FOLDS_PATH are where the fold file goes. Locally that is
   ../data/folds.csv, deliberately outside derived/: this file is committed to
   git, because its exact contents must survive unchanged for the life of the
   project and any change to it should show up in a diff. On Kaggle it can only
   be written to /kaggle/working/.

Settings
--------
7. N_FOLDS is the number of folds the data is split into.
8. SEEDS are the random seeds to try. The splitter cannot satisfy grouping and
   balance perfectly at the same time, and different seeds land differently,
   so a few are tried and the most even kept. Three is enough; chasing perfect
   balance is wasted effort.
9. INK_CUTOFF comes from Stage 2: slides with an ink score above it carry a
   visible pen mark.
"""

ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    DATA_DIR = "/kaggle/input/prostate-cancer-grade-assessment/"
    DERIVED_DIR = "/kaggle/working/derived/"
    FOLDS_DIR = "/kaggle/working/"
else:
    DATA_DIR = "../data/"
    DERIVED_DIR = os.path.join(DATA_DIR, "derived/")
    FOLDS_DIR = DATA_DIR


def find_input(name, required=True, local_dir=None):
    """Return the path of a file made earlier, or None if optional and missing."""
    candidates = [os.path.join(local_dir or DERIVED_DIR, name)]
    if ON_KAGGLE:
        for subfolder in ("", "derived", "data"):
            candidates += sorted(glob.glob(os.path.join("/kaggle/input", "*", subfolder, name)))
    for path in candidates:
        if os.path.exists(path):
            print(f"{name}: {path}")
            return path
    if required:
        raise FileNotFoundError(f"{name} not found; run the notebook that makes it, "
                                f"or attach its output as a dataset on Kaggle")
    return None


INVENTORY_PATH = find_input("slide_inventory.parquet")
DUPLICATE_GROUPS_PATH = find_input("duplicate_groups.parquet")
COLOUR_STATS_PATH = find_input("slide_colour_stats.parquet")

FOLDS_PATH = os.path.join(FOLDS_DIR, "folds.csv")

N_FOLDS = 5
SEEDS = [0, 1, 2]
INK_CUTOFF = 0.06

slide_inventory.parquet: ../data/derived/slide_inventory.parquet
duplicate_groups.parquet: ../data/derived/duplicate_groups.parquet
slide_colour_stats.parquet: ../data/derived/slide_colour_stats.parquet


## 1. Build the table to split

Three files, joined on `slide_id`: the inventory from Stage 1 (grade and
hospital), the duplicate groups from Stage 3, and the colour statistics from
Stage 2 (used only for the pen check at the end, never for splitting).

In [3]:
"""Join the three inputs into one table, one row per slide.

1. From the inventory, keep only what the split needs: slide_id, grade and
   hospital.
2. From the colour statistics, keep only slide_id and the ink score.
3. Join all three on slide_id. how="left" keeps every inventory slide even if
   it were missing from another table, so a gap shows up as an empty value
   instead of a vanished row.
4. A join is where rows quietly disappear or get duplicated, so check rather
   than assume: exactly 10,616 rows, every slide has a duplicate group and an
   ink score, and no slide appears twice.
5. has_pen marks slides whose ink score is above the cutoff.
6. Print the totals and show the first rows as a check.
"""

inventory = pd.read_parquet(INVENTORY_PATH)[["slide_id", "isup_grade", "provider"]]
duplicate_groups = pd.read_parquet(DUPLICATE_GROUPS_PATH)
colour_stats = pd.read_parquet(COLOUR_STATS_PATH)[["slide_id", "ink_frac"]]

slides = inventory.merge(duplicate_groups, on="slide_id", how="left")
slides = slides.merge(colour_stats, on="slide_id", how="left")

assert len(slides) == len(inventory) == 10616, "row count changed during the join"
assert slides["duplicate_group"].notna().all(), "a slide has no duplicate group"
assert slides["ink_frac"].notna().all(), "a slide has no ink score"
assert slides["slide_id"].is_unique, "duplicate slide_id"

slides["has_pen"] = slides["ink_frac"] > INK_CUTOFF
print(f"{len(slides)} slides, {slides['duplicate_group'].nunique()} duplicate groups, "
      f"{slides['has_pen'].sum()} with pen")
slides.head()

10616 slides, 10334 duplicate groups, 317 with pen


,slide_id,isup_grade,provider,duplicate_group,is_duplicate,ink_frac,has_pen
0,0005f7aaab2800f6170c399693a96917,0,karolinska,0,False,0.0,False
1,000920ad0b612851f8e01bcc880d9b3d,0,karolinska,1,False,0.0,False
2,0018ae58b01bdadc8e347995b69f99aa,4,radboud,2,False,0.0,False
3,001c62abd11fa4b57bf7a6c603a11bb9,4,karolinska,3,False,0.0,False
4,001d865e65ef5d2579c190a0e0350d8f,0,karolinska,4,False,0.0,False


## 2. The stratification key

We want folds balanced on grade *and* on hospital at once. The splitter only
takes one label to balance on, so we glue the two together into a single key:
a Radboud slide of grade 3 gets the key `3_radboud`. Balancing on that balances
both at the same time.

Twelve combinations, six grades times two hospitals.

In [4]:
"""Build the stratification key and count how many slides share each value.

The key joins grade and hospital with an underscore, e.g. "3_radboud", so that
balancing the folds on this one key balances both at once. value_counts counts
the slides per key, and sort_index lists the keys in order.
"""

slides["strat_key"] = (
    slides["isup_grade"].astype(str) + "_" + slides["provider"].astype(str)
)
print(slides["strat_key"].value_counts().sort_index().to_string())

strat_key
0_karolinska    1925
0_radboud        967
1_karolinska    1814
1_radboud        852
2_karolinska     668
2_radboud        675
3_karolinska     317
3_radboud        925
4_karolinska     481
4_radboud        768
5_karolinska     251
5_radboud        973


## 3. Split, and pick the most even of a few tries

`StratifiedGroupKFold` gets three things: the rows, the key to balance on, and
the group each row belongs to. It guarantees the grouping — a group never
straddles two folds — and does its best on the balance.

To compare seeds we need one number for "how even is this". We use the worst
case: for every grade and every hospital, how far that fold's share drifts from
the share in the dataset as a whole. Lower is better. The worst case is the
right thing to look at because a single badly-off fold is exactly what makes
one fold's score incomparable with the others.

In [5]:
"""Split the slides into folds with a few seeds, and keep the most even split.

1. assign_folds does one split for one seed (see its description below).
2. worst_imbalance scores how uneven a split is (see its description below).
3. Try every seed in SEEDS, and print each one's score.
4. Keep the seed with the lowest score and store its fold numbers in the
   table. If two seeds tie, the first one tried wins, so the choice is always
   the same.
"""


def assign_folds(slides, seed):
    """Return a fold number from 0 to N_FOLDS - 1 for every row.

    1. Create the splitter. shuffle=True with a fixed random_state means the
       split is random but the same every time for the same seed.
    2. Start every slide at -1, meaning "not yet assigned".
    3. split() is given the rows, the key to balance on, and each row's
       duplicate group. It guarantees a group never straddles two folds, and
       balances the key as well as it can. It yields one (train, validation)
       split per fold.
    4. Every slide is in exactly one fold's validation set, and that fold is
       the number it gets.
    5. Check that no slide was left at -1.
    """
    splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    fold = np.full(len(slides), -1, dtype=np.int8)
    for fold_number, (_, val_idx) in enumerate(
        splitter.split(slides, slides["strat_key"], groups=slides["duplicate_group"])
    ):
        fold[val_idx] = fold_number
    assert (fold >= 0).all(), "some slide was never assigned"
    return fold


def worst_imbalance(slides, fold):
    """Largest drift of any grade's or hospital's share, in any fold.

    1. For grade, then for hospital, work out each value's share of the whole
       dataset.
    2. crosstab with normalize="index" gives each value's share within each
       fold. reindex puts the columns in the same order as the overall shares,
       so the two line up.
    3. Subtract, take absolute values, and keep the largest drift found in
       either table. Lower is better. The worst case is what matters, because a
       single badly-off fold is exactly what makes its score incomparable with
       the others.
    """
    worst = 0.0
    for column in ("isup_grade", "provider"):
        overall = slides[column].value_counts(normalize=True)
        by_fold = (
            pd.crosstab(fold, slides[column], normalize="index")
            .reindex(columns=overall.index)
        )
        worst = max(worst, float((by_fold - overall).abs().to_numpy().max()))
    return worst


results = {}
for seed in SEEDS:
    fold = assign_folds(slides, seed)
    results[seed] = (fold, worst_imbalance(slides, fold))
    print(f"seed {seed}: worst share drift {results[seed][1]:.4f}")

best_seed = min(results, key=lambda s: results[s][1])
slides["fold"] = results[best_seed][0]
print(f"\nusing seed {best_seed}")

seed 0: worst share drift 0.0005
seed 1: worst share drift 0.0006
seed 2: worst share drift 0.0006

using seed 0


## 4. The checks that have to pass

The grouping assertion is the one that matters. If a duplicate group is split
across two folds, every score computed against this file is inflated and
nothing will tell you. Stop here if it fails — do not write the file.

In [6]:
"""The checks that must pass before the file is written.

1. For each duplicate group, count how many different folds its slides landed
   in. Any group with more than one is a leak, and the assert stops the
   notebook before anything is saved.
2. Every slide has a fold number between 0 and N_FOLDS - 1.
3. All 10,616 slides are present, and each appears once.
4. Print the number of slides in each fold.
"""

folds_per_group = slides.groupby("duplicate_group")["fold"].nunique()
leaked = folds_per_group[folds_per_group > 1]
assert len(leaked) == 0, f"{len(leaked)} duplicate groups straddle folds"

assert slides["fold"].between(0, N_FOLDS - 1).all()
assert len(slides) == 10616
assert slides["slide_id"].nunique() == 10616

print("no duplicate group straddles a fold")
print("\nslides per fold:")
print(slides["fold"].value_counts().sort_index().to_string())

no duplicate group straddles a fold

slides per fold:
fold
0    2123
1    2123
2    2123
3    2124
4    2123


## 5. Read the balance tables

Each column should be roughly constant down the rows. A grade whose share
swings from 0.08 in one fold to 0.15 in another means that fold's score will
differ from the others for reasons unrelated to the model.

The hospital table is the one most likely to be uneven. Duplicates are ten
times more common in Radboud, and grouped slides cannot be separated, so the
splitter has less room to move there.

In [7]:
"""The two balance tables: grade share by fold, and hospital share by fold.

1. crosstab counts slides by fold and grade (then by fold and hospital), and
   normalize="index" turns each fold's counts into shares that add up to 1.
2. Below each table, one extra row shows the share in the whole dataset, so
   every fold can be compared with it directly.
3. Shares are rounded to 4 decimal places.
"""

grade_by_fold = pd.crosstab(slides["fold"], slides["isup_grade"], normalize="index")
provider_by_fold = pd.crosstab(slides["fold"], slides["provider"], normalize="index")

print("grade share by fold (overall share in the last row)")
print(pd.concat([
    grade_by_fold,
    slides["isup_grade"].value_counts(normalize=True).sort_index().to_frame("overall").T,
]).round(4).to_string())

print("\nhospital share by fold")
print(pd.concat([
    provider_by_fold,
    slides["provider"].value_counts(normalize=True).to_frame("overall").T,
]).round(4).to_string())

grade share by fold (overall share in the last row)
isup_grade       0       1       2       3       4       5
0           0.2723  0.2515  0.1267  0.1173  0.1173  0.1149
1           0.2727  0.2511  0.1262  0.1168  0.1178  0.1154
2           0.2723  0.2511  0.1267  0.1168  0.1178  0.1154
3           0.2721  0.2514  0.1266  0.1168  0.1177  0.1153
4           0.2727  0.2506  0.1262  0.1173  0.1178  0.1154
overall     0.2724  0.2511  0.1265  0.1170  0.1177  0.1153

hospital share by fold
provider  karolinska  radboud
0             0.5144   0.4856
1             0.5134   0.4866
2             0.5144   0.4856
3             0.5137   0.4863
4             0.5139   0.4861
overall       0.5139   0.4861


## 6. Are the pen-marked slides spread out?

317 slides carry visible pen, all of them Radboud, and Stage 2 found the ink is
not evenly spread across grades. Stage 12 has to report a score on those slides
separately, which only works if every fold has some. Piled into one fold, that
check becomes impossible.

This is a diagnostic, not a constraint — we are not splitting on it.

In [8]:
"""Count the pen-marked slides in each fold, and compare with an even spread,
which is the total divided by the number of folds."""

print("pen-marked slides per fold:")
print(slides[slides["has_pen"]]["fold"].value_counts().sort_index().to_string())
print(f"\nexpected if evenly spread: {slides['has_pen'].sum() / N_FOLDS:.0f} per fold")

pen-marked slides per fold:
fold
0    66
1    62
2    64
3    61
4    64

expected if evenly spread: 63 per fold


## 7. Write the file, once

The notebook refuses to overwrite an existing fold assignment. That is
deliberate. The one way this file causes harm is being quietly regenerated
after results have been recorded against the old one, and a `to_csv` that
happily overwrites is all it takes.

To replace it on purpose you have to delete it by hand, which is enough of a
speed bump to make you think about the results you are invalidating.

In [9]:
"""Write the fold file, but never over an existing one.

1. Keep the five columns the file holds: slide_id, grade, hospital, duplicate
   group and fold number.
2. Look for a fold file that already exists: locally at ../data/folds.csv; on
   Kaggle also in any attached dataset, since a fresh Kaggle session starts
   with an empty working folder and would otherwise not know one exists.
3. If one exists, do not write anything. Compare it with what this run
   produced, value by value rather than type by type: reading a CSV back
   turns the int8 fold numbers into int64, so a type-by-type comparison would
   report a difference on every rerun, and a warning that always fires is one
   people stop reading. Print whether the two match. If they do not, the file
   on disk is the real one.
4. If none exists, create the folder if needed and write the file. On Kaggle,
   print a reminder that the copy committed to git is the one that counts, and
   that this file should be attached to later notebooks rather than
   regenerated.

To replace the fold file on purpose, delete it by hand.
"""

fold_table = slides[["slide_id", "isup_grade", "provider", "duplicate_group", "fold"]]

existing_path = find_input("folds.csv", required=False, local_dir=FOLDS_DIR)

if existing_path is not None:
    existing = pd.read_csv(existing_path)
    fresh = fold_table.reset_index(drop=True)
    same = (
        list(existing.columns) == list(fresh.columns)
        and len(existing) == len(fresh)
        and all((existing[col].to_numpy() == fresh[col].to_numpy()).all() for col in fresh.columns)
    )
    print(f"{existing_path} already exists — not overwriting.")
    print("the file on disk matches what this run produced" if same
          else "WARNING: the file on disk DIFFERS from what this run produced. "
               "The file on disk is the real one; do not replace it.")
else:
    os.makedirs(FOLDS_DIR, exist_ok=True)
    fold_table.to_csv(FOLDS_PATH, index=False)
    print(f"wrote {len(fold_table)} rows to {FOLDS_PATH}")
    print(f"seed {best_seed}, {N_FOLDS} folds, grouped by duplicate_group, "
          f"stratified by grade and hospital")
    if ON_KAGGLE:
        print("note: the folds.csv committed to git is the authoritative copy. "
              "Attach it to later notebooks rather than regenerating it.")

folds.csv: ../data/folds.csv
../data/folds.csv already exists — not overwriting.
the file on disk matches what this run produced


## What now exists

`data/folds.csv` — one row per slide, all 10,616, with the grade, the hospital,
the duplicate group and a fold number 0 to 4. Committed to git.

**This file is never regenerated.** Changing it makes every result recorded
against it incomparable with every result recorded after. If it ever seems to
need changing, that is a conversation, not a rerun.